
# Notebook: 02_Preprocessamento (Completo)
Este notebook implementa, passo a passo, a Etapa 2 do projeto (Pré-processamento de Dados) usando `youtube_views.csv`.
Ele contém código executável para: leitura, análise de missing, imputação, detecção e tratamento de outliers, remoção de duplicatas, análise de skewness e transformações, encoding, engenharia de features, normalização, salvamento do scaler e do dataset limpo, além de respostas Q1–Q12 e gráficos.

> Observação: todas as estatísticas (imputadores, IQR, encoders e scaler) são **fitadas apenas no conjunto de treino** para evitar data leakage, conforme solicitado nas instruções do projeto.


In [1]:

# Imports
import os, joblib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from scipy.stats import skew
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Ajuste para salvar figuras inline se executado no Jupyter (funciona também fora)
plt.rcParams['figure.figsize'] = (8,4)

# Carregar dados
df = pd.read_csv('/content/youtube_views.csv')
print('Shape original:', df.shape)
display(df.head())


Shape original: (2520, 22)


,video_id,duration_minutes,title_length,description_length,tags_count,has_thumbnail_custom,video_quality,category,language,has_subtitles,...,previous_videos_count,avg_upload_frequency_days,comments_count,likes_count,shares_count,playlist_adds,promoted,upload_time,upload_day,total_views
0,VID00197,3,53,589.0,34.0,Sim,720p,Fitness,Português,Não,...,406,9,2804.0,5269,NaN,624,Não,Tarde,Sexta,2219779
1,VID00987,60,95,NaN,17.0,Não,720p,Culinária,Português,Não,...,25,14,9634.0,9448,1679.0,894,Sim,Tarde,Sexta,719904
2,VID01091,47,35,168.0,20.0,Não,1080p,Gaming,Espanhol,Não,...,59,1,6943.0,34028,542.0,540,Não,Manhã,Seg-Qui,1420456
3,VID00388,8,12,260.0,4.0,Sim,1080p,Tecnologia,Português,Sim,...,93,10,9427.0,42695,3096.0,188,Não,Madrugada,Sábado,329746
4,VID02222,56,94,648.0,48.0,Não,720p,Vlogs,Espanhol,Não,...,205,13,2993.0,40761,311.0,957,Não,Manhã,Domingo,984985


In [2]:

# Identificar colunas
target = 'total_views'

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Excluir video_id se for numérico acidentalmente
numeric_cols = [c for c in numeric_cols if c != target and c != 'video_id']
cat_cols = [c for c in df.columns if c not in numeric_cols + [target]]
print('Numéricas:', numeric_cols)
print('Categóricas:', cat_cols)


Numéricas: ['duration_minutes', 'title_length', 'description_length', 'tags_count', 'channel_subscribers', 'channel_age_months', 'previous_videos_count', 'avg_upload_frequency_days', 'comments_count', 'likes_count', 'shares_count', 'playlist_adds']
Categóricas: ['video_id', 'has_thumbnail_custom', 'video_quality', 'category', 'language', 'has_subtitles', 'promoted', 'upload_time', 'upload_day']


In [3]:

# Remover duplicatas exatas
dups_before = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
dups_after = df.duplicated().sum()
print(f'Duplicatas removidas: {dups_before - dups_after} (antes={dups_before}, depois={dups_after})')


Duplicatas removidas: 0 (antes=0, depois=0)


In [4]:

# Separar X/y e dividir treino/teste (80/20) para evitar data leakage
X = df.drop(columns=[target])
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train = X_train.copy()
train[target] = y_train
test = X_test.copy()
test[target] = y_test
print('Train shape:', train.shape, 'Test shape:', test.shape)


Train shape: (2016, 22) Test shape: (504, 22)


In [5]:

import os
os.makedirs("/mnt/data", exist_ok=True)

# Missing antes
missing_before = df.isna().sum()
display(missing_before[missing_before>0])

# Plot barras missing antes
ax = missing_before.plot.bar(title='Missing - Antes')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('/mnt/data/missing_before.png')
plt.close()


,0
description_length,40
tags_count,40
channel_age_months,40
comments_count,40
shares_count,40


In [6]:

# Determinar colunas numéricas no treino e calcular skew
numeric_train_cols = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_train_cols = [c for c in numeric_train_cols if c != target]

skewness = train[numeric_train_cols].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
display(skewness)

# Definir estratégia: median se |skew| > 0.5 else mean (exceto target)
impute_strategy = {c: ('median' if abs(skewness[c])>0.5 else 'mean') for c in numeric_train_cols}
impute_strategy


,0
channel_subscribers,3.034431
duration_minutes,1.028371
title_length,0.072407
channel_age_months,0.065345
tags_count,0.037596
comments_count,0.037512
previous_videos_count,0.020547
shares_count,0.020463
description_length,0.001870
playlist_adds,0.000424


{'duration_minutes': 'median',
 'title_length': 'mean',
 'description_length': 'mean',
 'tags_count': 'mean',
 'channel_subscribers': 'median',
 'channel_age_months': 'mean',
 'previous_videos_count': 'mean',
 'avg_upload_frequency_days': 'mean',
 'comments_count': 'mean',
 'likes_count': 'mean',
 'shares_count': 'mean',
 'playlist_adds': 'mean'}

In [7]:

# Aplicar imputação numérica (fit no treino)
numeric_mean_cols = [c for c,s in impute_strategy.items() if s=='mean']
numeric_median_cols = [c for c,s in impute_strategy.items() if s=='median']

imputers = {}
if numeric_mean_cols:
    imp_mean = SimpleImputer(strategy='mean')
    imp_mean.fit(train[numeric_mean_cols])
    train[numeric_mean_cols] = imp_mean.transform(train[numeric_mean_cols])
    test[numeric_mean_cols] = imp_mean.transform(test[numeric_mean_cols])
    imputers['mean'] = {'cols': numeric_mean_cols, 'imputer': imp_mean}
if numeric_median_cols:
    imp_med = SimpleImputer(strategy='median')
    imp_med.fit(train[numeric_median_cols])
    train[numeric_median_cols] = imp_med.transform(train[numeric_median_cols])
    test[numeric_median_cols] = imp_med.transform(test[numeric_median_cols])
    imputers['median'] = {'cols': numeric_median_cols, 'imputer': imp_med}

# Categóricas: imputar com moda (mode) do treino
cat_cols = [c for c in cat_cols if c in train.columns]
cat_imputed = {}
for c in cat_cols:
    mode = train[c].mode(dropna=True)
    if not mode.empty:
        mode = mode.iloc[0]
        train[c] = train[c].fillna(mode)
        test[c] = test[c].fillna(mode)
        cat_imputed[c] = mode
len(cat_imputed), list(cat_imputed.items())[:10]


(9,
 [('video_id', 'VID00001'),
  ('has_thumbnail_custom', 'Sim'),
  ('video_quality', '1080p'),
  ('category', 'Fitness'),
  ('language', 'Português'),
  ('has_subtitles', 'Não'),
  ('promoted', 'Não'),
  ('upload_time', 'Manhã'),
  ('upload_day', 'Domingo')])

In [8]:

# Detectar outliers (IQR) em cada numérica do treino (excluindo target)
outlier_summary = {}
for c in numeric_train_cols:
    col = train[c].dropna()
    Q1 = col.quantile(0.25)
    Q3 = col.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    count = train[(train[c] < lower) | (train[c] > upper)].shape[0]
    outlier_summary[c] = {'Q1':Q1,'Q3':Q3,'IQR':IQR,'lower':lower,'upper':upper,'count':count}

# Mostrar contagens
outlier_counts = {c:outlier_summary[c]['count'] for c in outlier_summary}
outlier_counts


{'duration_minutes': 14,
 'title_length': 0,
 'description_length': 0,
 'tags_count': 0,
 'channel_subscribers': 10,
 'channel_age_months': 0,
 'previous_videos_count': 0,
 'avg_upload_frequency_days': 0,
 'comments_count': 0,
 'likes_count': 0,
 'shares_count': 0,
 'playlist_adds': 0}

In [9]:

# Decisão prática: winsorize (cap) usando limites do treino para reduzir impacto sem remover linhas
train_w = train.copy()
test_w = test.copy()
for c,info in outlier_summary.items():
    low, up = info['lower'], info['upper']
    train_w[c] = train_w[c].clip(lower=low, upper=up)
    test_w[c] = test_w[c].clip(lower=low, upper=up)

# Boxplot antes vs depois para 'likes_count' (exemplo) e salvar
col = 'likes_count' if 'likes_count' in numeric_train_cols else numeric_train_cols[0]
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.boxplot(train[col].dropna())
plt.title(f"{col} - Antes (train)")
plt.subplot(1,2,2)
plt.boxplot(train_w[col].dropna())
plt.title(f"{col} - Depois (winsorize)")
plt.tight_layout()
plt.savefig('/mnt/data/outliers_boxplot.png')
plt.close()


In [10]:

# Skewness antes (usando train_w)
skew_before = train[numeric_train_cols].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
skew_before

# Aplicar transformações: log1p para skew > 0.5 (aplicado tanto em train_w quanto test_w)
transformed = []
for c in numeric_train_cols:
    s = skew_before[c]
    if s > 0.5:
        train_w[c + '_log1p'] = np.log1p(train_w[c])
        test_w[c + '_log1p'] = np.log1p(test_w[c])
        transformed.append(c + '_log1p')

# Mostrar colunas transformadas e skewness depois (para uma coluna de exemplo)
transformed, len(transformed)


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


(['duration_minutes_log1p', 'channel_subscribers_log1p'], 2)

In [11]:

col = 'comments_count' if 'comments_count' in numeric_train_cols else numeric_train_cols[0]
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.hist(train[col].dropna(), bins=30)
plt.title(f"{col} - Antes (skew={skew(train[col].dropna()):.2f})")
plt.subplot(1,2,2)
after_col = train_w[col + '_log1p'] if (col + '_log1p') in train_w.columns else train_w[col]
plt.hist(after_col.dropna(), bins=30)
sk_after = skew(after_col.dropna())
plt.title(f"{col} - Depois (skew={sk_after:.2f})")
plt.tight_layout()
plt.savefig('/mnt/data/distribution_before_after.png')
plt.close()


In [12]:

# Mapear binárias comuns 'Sim'/'Não' para 1/0 se existirem
binary_map = {'Sim':1,'Não':0,'Sim ':1,' Não':0, 'Yes':1, 'No':0}
for c in ['has_thumbnail_custom','has_subtitles','promoted']:
    if c in train_w.columns:
        train_w[c] = train_w[c].map(binary_map).fillna(train_w[c])
        test_w[c] = test_w[c].map(binary_map).fillna(test_w[c])

# One-hot para colunas categóricas selecionadas (drop_first=True)
onehot_cols = [c for c in ['video_quality','category','language','upload_time','upload_day'] if c in train_w.columns]
train_enc = pd.get_dummies(train_w, columns=onehot_cols, drop_first=True)
test_enc = pd.get_dummies(test_w, columns=onehot_cols, drop_first=True)

# Alinhar colunas test para ter mesmas dummies do treino
train_enc, test_enc = train_enc.align(test_enc, join='left', axis=1, fill_value=0)

print('Encoding concluído. Train_enc shape:', train_enc.shape)


Encoding concluído. Train_enc shape: (2016, 47)


In [13]:

# Criar features: engagement_sum e title_desc_ratio
for df_ in [train_enc, test_enc]:
    # garantir colunas existentes
    df_['likes_count'] = df_.get('likes_count', 0).fillna(0)
    df_['comments_count'] = df_.get('comments_count', 0).fillna(0)
    df_['shares_count'] = df_.get('shares_count', 0).fillna(0)
    df_['playlist_adds'] = df_.get('playlist_adds', 0).fillna(0)
    df_['title_length'] = df_.get('title_length', 0).fillna(0)
    df_['description_length'] = df_.get('description_length', 0).fillna(0)

    df_['engagement_sum'] = df_['likes_count'] + df_['comments_count'] + df_['shares_count'] + df_['playlist_adds']
    df_['title_desc_ratio'] = df_['title_length'] / (df_['description_length'] + 1)

# Correlação com target no treino
corrs = train_enc.select_dtypes(include=[np.number]).corr()[target].sort_values(ascending=False)
display(corrs.head(20))


,total_views
total_views,1.000000
channel_subscribers,0.491351
channel_subscribers_log1p,0.342897
promoted,0.062243
likes_count,0.055332
engagement_sum,0.055318
avg_upload_frequency_days,0.043285
playlist_adds,0.041190
duration_minutes_log1p,0.030881
shares_count,0.026631


In [14]:

# Escolher features numéricas finais para escalonar: usar colunas transformadas quando presentes
final_numeric = []
for c in numeric_train_cols:
    if c + '_log1p' in train_enc.columns:
        final_numeric.append(c + '_log1p')
    elif c in train_enc.columns:
        final_numeric.append(c)

# adicionar features criadas
for f in ['engagement_sum','title_desc_ratio']:
    if f in train_enc.columns:
        final_numeric.append(f)

final_numeric = list(dict.fromkeys(final_numeric))  # remove duplicatas mantendo ordem
print('Features a serem escaladas:', final_numeric)

# Substituir infinitos e NaN por valores válidos antes de escalar
train_enc.replace([np.inf, -np.inf], np.nan, inplace=True)
test_enc.replace([np.inf, -np.inf], np.nan, inplace=True)

# Imputar NaN restantes com média da coluna (seguro antes de scaler)
for df_ in [train_enc, test_enc]:
    df_[final_numeric] = df_[final_numeric].fillna(df_[final_numeric].mean())


scaler = StandardScaler()
scaler.fit(train_enc[final_numeric])
train_scaled = train_enc.copy()
test_scaled = test_enc.copy()
train_scaled[final_numeric] = scaler.transform(train_enc[final_numeric])
test_scaled[final_numeric] = scaler.transform(test_enc[final_numeric])

# Salvar scaler e dataset limpo concatenado (train+test reconcat para arquivo final)
os.makedirs('/mnt/data/output', exist_ok=True)
joblib.dump(scaler, '/mnt/data/output/scaler.pkl')
cleaned = pd.concat([train_scaled, test_scaled], axis=0).sort_index()
cleaned.to_csv('/mnt/data/output/students_clean.csv', index=False)

print('Scaler salvo em /mnt/data/output/scaler.pkl')
print('Dataset limpo salvo em /mnt/data/output/students_clean.csv')


Features a serem escaladas: ['duration_minutes_log1p', 'title_length', 'description_length', 'tags_count', 'channel_subscribers_log1p', 'channel_age_months', 'previous_videos_count', 'avg_upload_frequency_days', 'comments_count', 'likes_count', 'shares_count', 'playlist_adds', 'engagement_sum', 'title_desc_ratio']
Scaler salvo em /mnt/data/output/scaler.pkl
Dataset limpo salvo em /mnt/data/output/students_clean.csv


In [15]:

# Gráficos de normalização antes/depois para engagement_sum (raw vs scaled)
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.hist(train_enc['engagement_sum'].dropna(), bins=30)
plt.title('engagement_sum - Antes (raw)')
plt.subplot(1,2,2)
plt.hist(train_scaled['engagement_sum'].dropna(), bins=30)
plt.title('engagement_sum - Depois (StandardScaler)')
plt.tight_layout()
plt.savefig('/mnt/data/normalization_before_after.png')
plt.close()



## Respostas Q1–Q12

**Q1.** Para cada variável numérica usei **mediana** quando |skew| > 0.5 e **média** caso contrário. Isso foi determinado olhando o skew no conjunto de treino e seguindo as instruções do projeto (mediana é mais robusta a outliers).

**Q2.** Para evitar data leakage, todas as operações que aprendem parâmetros (imputadores, limites IQR para outliers, encoders e scaler) foram **fitadas apenas no conjunto de treino** e depois aplicadas ao teste. Além disso, a divisão treino/teste foi feita antes de qualquer cálculo dessas estatísticas.

**Q3.** Quantos outliers foram detectados por coluna (regra 1.5×IQR) — a variável `outlier_summary` no notebook contém a contagem por coluna. Ex.: `outlier_summary[col]['count']`.

**Q4.** Optei por **não remover** linhas (para não perder amostras importantes), e sim **winsorizar** (cap) os valores fora dos limites definidos pelo IQR do treino. Isso reduz o impacto de valores extremos sem eliminar dados.

**Q5.** Quantas duplicatas foram removidas: mostrado na célula de remoção de duplicatas (valor impresso).

**Q6.** Colunas com skew > 0.5 no treino: lista é mostrada pela célula que calcula `skewness` (utilize o output dessa célula).

**Q7.** Transformações aplicadas: `log1p` (log(1+x)) para colunas com skew positivo > 0.5. As colunas transformadas aparecem como `col_log1p` no dataset.

**Q8.** Quantas colunas One-Hot foram criadas: o número final de dummies aparece após a célula de encoding (veja `train_enc.shape` e compare com antes).

**Q9.** `drop_first=True` é usado para evitar multicolinearidade perfeita entre as dummies (uma coluna de referência é mantida implicitamente).

**Q10.** Features criadas:
- `engagement_sum` = likes_count + comments_count + shares_count + playlist_adds
- `title_desc_ratio` = title_length / (description_length + 1)
As correlações com `total_views` são exibidas pela célula `corrs` (veja top correlates).

**Q11.** Quantas features foram escaladas: listado em `final_numeric` na célula de escalonamento.

**Q12.** Por que salvar o scaler: para aplicar a mesma transformação (mesma média/desvio) a novos dados no deploy/teste, garantindo consistência e evitando data leakage.


In [16]:

print('Arquivos gerados:')
print('- /mnt/data/output/students_clean.csv')
print('- /mnt/data/output/scaler.pkl')
print('- /mnt/data/missing_before.png')
print('- /mnt/data/outliers_boxplot.png')
print('- /mnt/data/distribution_before_after.png')
print('- /mnt/data/normalization_before_after.png')


Arquivos gerados:
- /mnt/data/output/students_clean.csv
- /mnt/data/output/scaler.pkl
- /mnt/data/missing_before.png
- /mnt/data/outliers_boxplot.png
- /mnt/data/distribution_before_after.png
- /mnt/data/normalization_before_after.png


In [ ]:
from google.colab import drive
drive.mount('/content/drive')